# Residual Stream Decomposition

Break down the residual stream by source: embedding, attention, MLP
contributions with norms, cosines, and unembed projection.

## Why This Matters

Composition residual stream decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_decomposition import (
    component_contribution_norms, cumulative_component_balance,
    residual_component_cosines, residual_projection_decomposition,
    residual_stream_decomposition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Component Contribution Norms

How large is each component's output at each layer?

In [ ]:
result = component_contribution_norms(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn={p['attn_norm']:.4f}, "
          f"mlp={p['mlp_norm']:.4f}, resid={p['residual_norm']:.4f}")

## Cumulative Component Balance

Embed vs attention vs MLP contribution fractions over layers.

In [ ]:
result = cumulative_component_balance(model, tokens, position=-1)
print(f"Embed norm: {result['embed_norm']:.4f}")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: embed={p['embed_fraction']:.3f}, "
          f"attn={p['attn_fraction']:.3f}, mlp={p['mlp_fraction']:.3f}")

## Component-Residual Cosines

How aligned is each component with the overall residual?

In [ ]:
result = residual_component_cosines(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn_cos={p['attn_cosine']:.4f}, "
          f"mlp_cos={p['mlp_cosine']:.4f}")

## Unembed Projection Decomposition

In [ ]:
result = residual_projection_decomposition(model, tokens, position=-1)
print(f"Target token: {result['target_token']}")
for p in result['per_layer']:
    bar = '█' * int(p['projection_fraction'] * 40)
    print(f"  Layer {p['layer']}: proj={p['unembed_projection']:.4f}, "
          f"frac={p['projection_fraction']:.3f} {bar}")

## Summary

In [ ]:
result = residual_stream_decomposition_summary(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn_norm={p['attn_norm']:.4f}, "
          f"attn_cos={p['attn_cosine']:.4f}, mlp_cos={p['mlp_cosine']:.4f}")